# GraviText - Tokenization, embeddings and transformers

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil*

In [ ]:
#r "../src/GraviNum/bin/Release/net10.0/Gravicode.Science.GraviNum.dll"
#r "../src/GraviFrame/bin/Release/net10.0/Gravicode.Science.GraviFrame.dll"
#r "../src/GraviLearn/bin/Release/net10.0/Gravicode.Science.GraviLearn.dll"
#r "../src/GraviText/bin/Release/net10.0/Gravicode.Science.GraviText.dll"
#r "nuget: ScottPlot, 5.1.59"

using Gravicode.Science.GraviFrame;
using Gravicode.Science.GraviLearn.Decomposition;
using Gravicode.Science.GraviNum;
using Gravicode.Science.GraviText.Embeddings;
using Gravicode.Science.GraviText.Linguistics;
using Gravicode.Science.GraviText.Tasks;
using Gravicode.Science.GraviText.Tokenization;
using Gravicode.Science.GraviText.Transformers;
using Gravicode.Science.GraviText.Vectorization;

var reviews = DataFrame.ReadCsv("../datasets/imdb_reviews.csv");
var documents = Enumerable.Range(0, reviews.RowCount).Select(i => reviews.Text("review")[i]!).ToArray();
var labels = Enumerable.Range(0, reviews.RowCount).Select(i => reviews.Text("sentiment")[i]!).ToArray();
Console.WriteLine($"{documents.Length} reviews");

## Tokenization and stemming, in both languages

In [ ]:
Console.WriteLine(string.Join(" | ", new RegexTokenizer().Tokenize("Gravicode Studios membangun AI di .NET!")));

foreach (var w in new[] { "connection", "running", "ponies" })
    Console.WriteLine($"en {w,-14}-> {PorterStemmer.Stem(w)}");
foreach (var w in new[] { "makanan", "membaca", "berlari" })
    Console.WriteLine($"id {w,-14}-> {IndonesianStemmer.Stem(w)}");

## Sentiment analysis

In [ ]:
var rng = new GraviRandom(42);
var order = rng.Permutation(documents.Length);
var cut = (int)(documents.Length * 0.75);

var trainDocs = order.Take(cut).Select(i => documents[i]).ToArray();
var trainLabels = order.Take(cut).Select(i => labels[i]).ToArray();
var testDocs = order.Skip(cut).Select(i => documents[i]).ToArray();
var testLabels = order.Skip(cut).Select(i => labels[i]).ToArray();

var classifier = new TextClassifier().Train(trainDocs, trainLabels);
Console.WriteLine($"held-out accuracy: {classifier.Evaluate(testDocs, testLabels):P2}");

foreach (var label in classifier.Labels)
    Console.WriteLine($"{label,-10}{string.Join(", ", classifier.TopFeatures(label, 8).Select(t => t.Term))}");

## Word embeddings and a t-SNE projection

Raw vectors share a large common direction; removing it is what makes the cosine informative on a small corpus.

In [ ]:
var tokenized = documents.Select(d => new RegexTokenizer().Tokenize(d)).ToList();
var raw = new Word2Vec(dimensions: 64, windowSize: 4, minCount: 2, epochs: 25, seed: 42).Train(tokenized);
var embeddings = raw.RemoveCommonComponent();

var probe = raw.Words.Take(40).ToArray();
Console.WriteLine($"mean pairwise cosine: {probe.SelectMany(a => probe.Where(b => b != a).Select(b => raw.Similarity(a, b))).Average():F3} raw");
Console.WriteLine($"                      {probe.SelectMany(a => probe.Where(b => b != a).Select(b => embeddings.Similarity(a, b))).Average():F3} centred");

In [ ]:
var vocabulary = embeddings.Words.Where(w => w.Length > 3).Take(90).ToArray();
var vectors = NdArray.Zeros(vocabulary.Length, embeddings.Dimensions);
for (var i = 0; i < vocabulary.Length; i++)
{
    var v = embeddings[vocabulary[i]];
    for (var d = 0; d < embeddings.Dimensions; d++) vectors[i, d] = v.At(d);
}

var projection = new TStochasticNeighborEmbedding(2, perplexity: 12, iterations: 400, seed: 42).FitTransform(vectors);

var plot = new ScottPlot.Plot();
var xs = Enumerable.Range(0, vocabulary.Length).Select(i => projection[i, 0]).ToArray();
var ys = Enumerable.Range(0, vocabulary.Length).Select(i => projection[i, 1]).ToArray();
var scatter = plot.Add.ScatterPoints(xs, ys);
scatter.MarkerSize = 8;
for (var i = 0; i < vocabulary.Length; i += 3)
    plot.Add.Text(vocabulary[i], xs[i], ys[i]).LabelFontSize = 9;

plot.Title("Word vectors projected with t-SNE");
plot.GetPngHtml(950, 750)

## Transformer encoder

**No pretrained weights are bundled.** The forward pass is complete and correct, but a fresh model is randomly initialised, so its vectors are structurally valid rather than semantically meaningful. Call LoadWeights before treating the output as embeddings.

In [ ]:
var vocabulary2 = WordPieceTokenizer.Train(documents, vocabularySize: 900, minFrequency: 2);
var tokenizer = new WordPieceTokenizer(vocabulary2);
var config = new TransformerConfig(vocabulary2.Count, HiddenSize: 128, Layers: 4, Heads: 8, IntermediateSize: 512, MaxPositions: 128);
var model = new TransformerModel(config, seed: 42).WithTokenizer(tokenizer);

Console.WriteLine(model);
model.Forward(new[] { 1, 2, 3, 4, 5, 6 });

var attention = model.AttentionMaps[0][0];
var plot3 = new ScottPlot.Plot();
plot3.Add.Heatmap(attention.To2DArray());
plot3.Title("Layer 1, head 1 attention (rows sum to 1)");
plot3.GetPngHtml(600, 500)

## Byte-pair encoding

Start from characters, repeatedly merge the most frequent adjacent pair, record each merge in order.
Applying it later means replaying those merges.

The **merge order is the model** — the same token set applied in a different order produces a
different segmentation, which is why `Save` writes ranked merges rather than a vocabulary.


In [ ]:
var bpeCorpus = new List<string>();
foreach (var (word, count) in new[] { ("low", 5), ("lower", 2), ("newest", 6), ("widest", 3) })
    for (var i = 0; i < count; i++) bpeCorpus.Add(word);

var bpe = BpeTokenizer.Train(bpeCorpus, vocabularySize: 60, minFrequency: 1);

Console.WriteLine($"{bpe.Merges.Count} merges learned. The first few:");
foreach (var (left, right) in bpe.Merges.Take(4))
    Console.WriteLine($"  {left} + {right} -> {left + right}");

Console.WriteLine($"\n'newest' -> [{string.Join(", ", bpe.Encode("newest"))}]   (frequent, so one token)");
Console.WriteLine($"'lowest' -> [{string.Join(", ", bpe.Encode("lowest"))}]   (never seen, but decomposes)");
Console.WriteLine($"round trip: '{bpe.Decode(bpe.Encode("lowest"))}'");


## Unigram (SentencePiece)

A different algorithm, not a variant. Start from a large candidate vocabulary, give every piece a
probability, and prune downwards. Segmentation is then Viterbi over the piece lattice — globally
optimal rather than greedy.

Because every piece carries a probability, alternative segmentations can be **sampled**. That is
subword regularisation, and BPE cannot do it at all.


In [ ]:
string[] unigramCorpus =
[
    "the cat sat on the mat", "the dog sat on the log", "the cat and the dog",
    "a cat on a mat", "the mat and the log",
];

var unigram = UnigramTokenizer.Train(unigramCorpus, vocabularySize: 80, seedSize: 400);

Console.WriteLine($"{unigram.PieceCount} pieces after pruning");
Console.WriteLine($"'the cat sat' -> [{string.Join(", ", unigram.Encode("the cat sat"))}]");
Console.WriteLine($"round trip is exact: '{unigram.Decode(unigram.Encode("the cat sat"))}'");
Console.WriteLine("whitespace is ENCODED rather than split on, so decoding is plain concatenation\n");

var samplingRng = new GraviRandom(7);
var seen = new HashSet<string>(StringComparer.Ordinal);
for (var i = 0; i < 60; i++)
    seen.Add(string.Join(" | ", unigram.SampleEncoding("the cat sat", samplingRng, alpha: 0.2)));

Console.WriteLine($"{seen.Count} distinct segmentations of the same sentence:");
foreach (var segmentation in seen.Take(4)) Console.WriteLine($"  {segmentation}");


## CRF sequence labelling

Labelling each token independently produces sequences that are locally plausible and globally
impossible — an `I-PER` with no `B-PER` before it. A CRF adds transition scores and decodes the
best *sequence*.

`Forbid` gives a transition a score no path can recover from. A learned penalty can always be
outvoted by a confident emission; a forbidden one cannot.


In [ ]:
using Gravicode.Science.GraviText.Sequence;

var crfLabels = new[] { "O", "B-PER", "I-PER" };
var crf = new LinearChainCrf(crfLabels.Length);
crf.ApplyBioConstraints(crfLabels);

var wanted = NdArray.Full(-10.0, 2, crfLabels.Length);
wanted[0, 0] = 10;   // strongly wants O
wanted[1, 2] = 10;   // strongly wants I-PER

Console.WriteLine("Emissions want 'O' then 'I-PER', which the BIO scheme forbids:");
Console.WriteLine($"  independent argmax : O, I-PER");
Console.WriteLine($"  the CRF decodes    : {string.Join(", ", crf.Decode(wanted).Select(i => crfLabels[i]))}\n");

// Trained on alternating labels with NO emission signal at all, so anything it
// gets right came from the transition structure alone.
var alternating = Enumerable.Range(0, 40).Select(_ => NdArray.Zeros(6, 2)).ToList();
var alternatingTags = Enumerable.Range(0, 40).Select(_ => new[] { 0, 1, 0, 1, 0, 1 }).ToList();

var learnedCrf = new LinearChainCrf(2).Fit(alternating, alternatingTags, epochs: 150, learningRate: 0.5);
Console.WriteLine($"0->1 scores {learnedCrf.Transition(0, 1):F3}, 0->0 scores {learnedCrf.Transition(0, 0):F3}");
Console.WriteLine($"decoding featureless input: [{string.Join(", ", learnedCrf.Decode(NdArray.Zeros(6, 2)))}]");


## Trained NER

Feature-based tagging — word shape, affixes, capitalisation, neighbours — feeding a CRF. The
shape feature does most of the work: mapping capitals to `X` and lower-case to `x` turns "Jakarta"
and "Bandung" into the same `Xxxxxxx`, so evidence about one transfers to the other.

**Evaluate on entities, not tokens.** Token accuracy is dominated by the `O` tag.


In [ ]:
var annotated = TaggedSentence.LoadConll("../datasets/ner_conll.txt");
var trainCut = (int)(annotated.Count * 0.75);

var ner = new TrainedNer().Fit(annotated.Take(trainCut).ToList());
var heldOut = annotated.Skip(trainCut).ToList();

Console.WriteLine($"entity score  : {ner.Evaluate(heldOut)}");
Console.WriteLine($"token accuracy: {ner.TokenAccuracy(heldOut):P2}  <- dominated by 'O'\n");

Console.WriteLine("Names appearing NOWHERE in the corpus, recognised from shape and context:");
foreach (var sentence in new[]
{
    "Kartika Wijaya bekerja di Gravicode .",
    "Zulkarnain tinggal di Surabaya .",
})
{
    Console.WriteLine($"  \"{sentence}\"");
    foreach (var entity in ner.Recognize(sentence))
        Console.WriteLine($"      {entity.Type,-4} {entity.Text}");
}

Console.WriteLine("\nThe corpus is generated, so 98% F1 means the model learned the templates -");
Console.WriteLine("NOT that it would score 98% on newswire.");


## Decoder stack and causality

Causality is the one property you cannot see by reading generated text. Change the **last** token
and measure how far the earlier hidden states moved: without the mask a decoder trains beautifully
and generates nothing, having learned to read the future.


In [ ]:
using Gravicode.Science.GraviText.Generation;

var decoderVocab = new Vocabulary();
foreach (var word in "the cat sat on a mat dog log and ran".Split(' ')) decoderVocab.Add(word);

var decoderConfig = new TransformerConfig(
    VocabularySize: decoderVocab.Count, HiddenSize: 32, Layers: 2, Heads: 4,
    IntermediateSize: 64, MaxPositions: 32);

var decoder = new TransformerDecoder(decoderConfig, decoderVocab, new GraviRandom(5));

var stateA = decoder.Forward(new[] { 5, 6, 7, 8 });
var stateB = decoder.Forward(new[] { 5, 6, 7, 12 });

var maxDrift = 0.0;
for (var t = 0; t < 3; t++)
    for (var d = 0; d < stateA.Shape[1]; d++)
        maxDrift = Math.Max(maxDrift, Math.Abs(stateA[t, d] - stateB[t, d]));

Console.WriteLine($"max drift across positions 0-2 after changing the last token: {maxDrift:E2}");
Console.WriteLine($"perplexity on a short sequence: {decoder.Perplexity(new[] { 5, 6, 7, 8, 9 }):F2}");
Console.WriteLine($"a uniform guess over {decoderVocab.Count} tokens would score {decoderVocab.Count};");
Console.WriteLine("random weights do no better, and anything LOWER would mean something is leaking\n");

var generationRng = new GraviRandom(11);
Console.WriteLine($"greedy  : [{string.Join(", ", decoder.Generate(new[] { 5, 6 }, 6, SamplingOptions.Greedy))}]");
Console.WriteLine($"nucleus : [{string.Join(", ", decoder.Generate(new[] { 5, 6 }, 6, SamplingOptions.Nucleus, generationRng))}]");
Console.WriteLine("\nThe weights are random, so the tokens mean nothing - what this shows is that");
Console.WriteLine("the sampling machinery works, not that the model has anything to say.");


## Attention over a causal sequence

The attention matrix of a decoder is strictly lower-triangular by construction. Rendering it makes
the mask visible: everything above the diagonal is exactly zero, so no position can read its future.


In [ ]:
var causal = new CausalSelfAttention(decoderConfig, new GraviRandom(3));
causal.Forward(new GraviRandom(9).StandardNormal(10, 32));

var head = causal.LastAttention[0];
var weights = new double[10, 10];
for (var i = 0; i < 10; i++)
    for (var j = 0; j < 10; j++)
        weights[i, j] = head[i, j];

var attentionPlot = new ScottPlot.Plot();
var heatmap = attentionPlot.Add.Heatmap(weights);
heatmap.Colormap = new ScottPlot.Colormaps.Viridis();
attentionPlot.Title("Causal attention - nothing above the diagonal");
attentionPlot.XLabel("key position (attended to)");
attentionPlot.YLabel("query position");
attentionPlot.GetPngHtml(650, 550)


## Loading a pretrained checkpoint

`TransformerCheckpoint.Load` fills **every** parameter, not just the embedding tables. The gap it
closes was worse than it looked: `LoadOnnxWeights` loaded the embeddings only, so a model reporting
`HasPretrainedWeights == true` still ran on random attention weights.

No weights ship with this repository, so a small checkpoint is written here in PyTorch's
`(out, in)` orientation and loaded back. The end-to-end check lives in
`tools/verify/checkpoint_interop.py`, which runs the same encoder in NumPy and agrees to 2.6e-07.

In [ ]:
using Gravicode.Science.GraviNum.Io;

var checkpointConfig = new TransformerConfig(
    VocabularySize: 40, HiddenSize: 16, Layers: 2, Heads: 2,
    IntermediateSize: 32, MaxPositions: 24);

string WriteCheckpoint(int skipLayer)
{
    var builder = new OnnxGraphBuilder("input", checkpointConfig.HiddenSize);
    var weightRng = new GraviRandom(7);

    NdArray Noise(int rows, int columns)
    {
        var array = NdArray.Zeros(rows, columns);
        for (var i = 0; i < array.Size; i++) array.SetAt(i, weightRng.Normal() * 0.05);
        return array;
    }

    NdArray Constant(int size, double value)
    {
        var array = NdArray.Zeros(size);
        for (var i = 0; i < size; i++) array.SetAt(i, value);
        return array;
    }

    builder.AddInitializer("bert.embeddings.word_embeddings.weight",
        Noise(checkpointConfig.VocabularySize, checkpointConfig.HiddenSize));
    builder.AddInitializer("bert.embeddings.position_embeddings.weight",
        Noise(checkpointConfig.MaxPositions, checkpointConfig.HiddenSize));
    builder.AddInitializer("bert.embeddings.LayerNorm.weight", Constant(checkpointConfig.HiddenSize, 1.0));
    builder.AddInitializer("bert.embeddings.LayerNorm.bias", Constant(checkpointConfig.HiddenSize, 0.0));

    for (var layer = 0; layer < checkpointConfig.Layers; layer++)
    {
        if (layer == skipLayer) continue;
        var prefix = $"bert.encoder.layer.{layer}";

        foreach (var part in new[]
        {
            "attention.self.query", "attention.self.key",
            "attention.self.value", "attention.output.dense",
        })
        {
            builder.AddInitializer($"{prefix}.{part}.weight",
                Noise(checkpointConfig.HiddenSize, checkpointConfig.HiddenSize));
            builder.AddInitializer($"{prefix}.{part}.bias", Constant(checkpointConfig.HiddenSize, 0.0));
        }

        builder.AddInitializer($"{prefix}.attention.output.LayerNorm.weight", Constant(checkpointConfig.HiddenSize, 1.0));
        builder.AddInitializer($"{prefix}.attention.output.LayerNorm.bias", Constant(checkpointConfig.HiddenSize, 0.0));

        // PyTorch stores (out, in), so the expansion is (intermediate, hidden).
        builder.AddInitializer($"{prefix}.intermediate.dense.weight",
            Noise(checkpointConfig.IntermediateSize, checkpointConfig.HiddenSize));
        builder.AddInitializer($"{prefix}.intermediate.dense.bias", Constant(checkpointConfig.IntermediateSize, 0.0));
        builder.AddInitializer($"{prefix}.output.dense.weight",
            Noise(checkpointConfig.HiddenSize, checkpointConfig.IntermediateSize));
        builder.AddInitializer($"{prefix}.output.dense.bias", Constant(checkpointConfig.HiddenSize, 0.0));

        builder.AddInitializer($"{prefix}.output.LayerNorm.weight", Constant(checkpointConfig.HiddenSize, 1.0));
        builder.AddInitializer($"{prefix}.output.LayerNorm.bias", Constant(checkpointConfig.HiddenSize, 0.0));
    }

    var path = Path.Combine(Path.GetTempPath(), $"gravi-{Guid.NewGuid():N}.onnx");
    builder.AddNode("Identity", ["input"], "output");
    builder.Save(path, "output", checkpointConfig.HiddenSize);
    return path;
}

var checkpointPath = WriteCheckpoint(skipLayer: -1);

// Names are a convention and the file is the only authority on which one it follows.
foreach (var (name, shape) in TransformerCheckpoint.Inspect(checkpointPath).Take(4))
    Console.WriteLine($"  {name,-58} [{string.Join(", ", shape)}]");

var restored = new TransformerModel(checkpointConfig);
Console.WriteLine($"before: HasPretrainedWeights = {restored.HasPretrainedWeights}");
Console.WriteLine(TransformerCheckpoint.Load(restored, checkpointPath).ToString());
Console.WriteLine($"after : HasPretrainedWeights = {restored.HasPretrainedWeights}");

var restoredStates = restored.Forward([3, 9, 14, 2]);
Console.WriteLine($"{restoredStates.Shape[0]} x {restoredStates.Shape[1]} hidden states");

In [ ]:
// A wrong transpose. PyTorch's nn.Linear stores (out, in) and computes x W^T; DenseLayer stores
// (in, out). A 768x768 attention projection is square, so both readings are consistent and the
// mistake loads silently - the check is made against the non-square feed-forward weight.
try
{
    TransformerCheckpoint.Load(new TransformerModel(checkpointConfig), checkpointPath,
        CheckpointNames.HuggingFaceBert with { Transposed = false });
}
catch (InvalidDataException error) { Console.WriteLine($"refused: {error.Message}"); }

// A partial export. Lenient mode loads and names the gap; either way HasPretrainedWeights stays
// false, because a model with one of two layers loaded produces output that is neither the
// checkpoint's nor a random model's, and nothing downstream could tell.
var partialPath = WriteCheckpoint(skipLayer: 1);
var lenientModel = new TransformerModel(checkpointConfig);
var partialReport = TransformerCheckpoint.Load(lenientModel, partialPath, strict: false);

Console.WriteLine($"partial: {partialReport}");
Console.WriteLine($"HasPretrainedWeights stays {lenientModel.HasPretrainedWeights}");
Console.WriteLine($"first missing: {partialReport.Missing[0]}");

File.Delete(checkpointPath);
File.Delete(partialPath);